In [6]:
import pickle
import os
import json
from utils import answer_function

def produce_diffed_data(file1, file2):
    joint_data = []
    judge_wrong = []
    judge_right = []
    both_right = []
    both_wrong = []
    model_1, model_2 = file1['metadata']['model_name'], file2['metadata']['model_name']
    data_1, data_2 = file1['data'], file2['data']
    completes = 0
    for i in range(len(data_1['questions'])):
        assert data_1['questions'][i] == data_2['questions'][i], f"Questions do not match at index {i}"
        assert data_1['ground_truth_answers'][i] == data_2['ground_truth_answers'][i], f"Answers do not match at index {i}"
        completions_1, completions_2 = data_1['completions'][i], data_2['completions'][i]
        try:
            completions_1_clean = answer_function(model_1)(completions_1)
        except Exception as e:
            print(f"Error processing completion for model {model_1} at index {i}: {e}")
            print(f"Completion text: {completions_1}")
        try:
            completions_2_clean = answer_function(model_2)(completions_2)
        except Exception as e:
            print(f"Error processing completion for model {model_2} at index {i}: {e}")
            print(f"Completion text: {completions_2}")
        if completions_1_clean is None or completions_2_clean is None:
            continue
        completes += 1
        ea_1, ea_2 = data_1['extracted_answers'][i], data_2['extracted_answers'][i]
        s_1, s_2 = data_1['scores'][i], data_2['scores'][i]
        if s_1 != s_2:
            joint_data.append({
                'question': data_1['questions'][i],
                'ground_truth_answer': data_1['ground_truth_answers'][i],
                'judge_completion': completions_1_clean,
                'judge_extracted_answer': ea_1,
                'judge_score': s_1,
                'ref_completion': completions_2_clean,
                'ref_extracted_answer': ea_2,
                'ref_score': s_2,
            })
            if s_1 == 1 and s_2 == 0:
                judge_right.append(i)
            elif s_1 == 0 and s_2 == 1:
                judge_wrong.append(i)
        elif s_1 == 1 and s_2 == 1:
            both_right.append(i)
        elif s_1 == 0 and s_2 == 0:
            both_wrong.append(i)
        else:
            raise ValueError(f"Unexpected scores at index {i}: {s_1}, {s_2}")
    payload = {
        'metadata': {
            'judge': model_1,
            'ref': model_2,
            'total_examples': completes,
            'differing_examples': len(joint_data),
            'num_right': len(judge_right),
            'num_wrong': len(judge_wrong),
            'num_both_right': len(both_right),
            'num_both_wrong': len(both_wrong),
        },
        'legit_idx': judge_right,
        'illegit_idx': judge_wrong,
        'both_right_idx': both_right,
        'both_wrong_idx': both_wrong,
        'data': joint_data
    }
    return payload


for i, item in enumerate(sorted(os.listdir("math/"))):
    if os.path.isdir("math/" + item):
        continue

    with open("math/" + item, 'rb') as f:
        f1 = pickle.load(f)
    
    for j, item2 in enumerate(sorted(os.listdir("math/"))):
        if i >= j or os.path.isdir("math/" + item2):
            continue

        with open("math/" + item2, 'rb') as f:
            f2 = pickle.load(f)

        diffed_data = produce_diffed_data(f1, f2)
        out_filename = f"data_verifiable_preference/diffs/all_diffs.json"
        with open(out_filename, 'a') as out_f:
            out_f.write(json.dumps(diffed_data) + "\n")



In [8]:
full = []
with open(out_filename, "r") as f:
    for raw in f.readlines():
        full.append(json.loads(raw))

In [19]:
for item in full:
    print(json.dumps(item['metadata'], indent=2))
    print("---")

{
  "judge": "Qwen/Qwen3-0.6B",
  "ref": "Qwen/Qwen3-4B",
  "total_examples": 4396,
  "differing_examples": 1046,
  "num_right": 72,
  "num_wrong": 974,
  "num_both_right": 3223,
  "num_both_wrong": 127
}
---
{
  "judge": "Qwen/Qwen3-0.6B",
  "ref": "Qwen/Qwen3-8B",
  "total_examples": 4382,
  "differing_examples": 1012,
  "num_right": 40,
  "num_wrong": 972,
  "num_both_right": 3247,
  "num_both_wrong": 123
}
---
{
  "judge": "Qwen/Qwen3-0.6B",
  "ref": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
  "total_examples": 4532,
  "differing_examples": 1116,
  "num_right": 359,
  "num_wrong": 757,
  "num_both_right": 2965,
  "num_both_wrong": 451
}
---
{
  "judge": "Qwen/Qwen3-0.6B",
  "ref": "deepseek-r1-0528",
  "total_examples": 4532,
  "differing_examples": 1566,
  "num_right": 1242,
  "num_wrong": 324,
  "num_both_right": 2082,
  "num_both_wrong": 884
}
---
{
  "judge": "Qwen/Qwen3-0.6B",
  "ref": "google/gemma-3-12b-it",
  "total_examples": 4532,
  "differing_examples": 1080,
  "num_ri

In [3]:
print(answer_function('gemma')(results_gemma_12['data']['completions'][4532]))


Let $S = \csc (2^3)^\circ + \csc (2^4)^\circ + \csc (2^5)^\circ + \dots + \csc (2^{2019})^\circ$.
We are given that $S = \sec n^\circ$ for some positive integer $n$ such that $0 < n < 180$.
We have
\[ S = \sum_{k=3}^{2019} \csc (2^k)^\circ = \csc (2^3)^\circ + \csc (2^4)^\circ + \csc (2^5)^\circ + \dots + \csc (2^{2019})^\circ. \]
We know that $\csc x = \frac{1}{\sin x}$.
We are given that $S = \sec n^\circ = \frac{1}{\cos n^\circ}$.
We have the identity $\csc x = \cot \frac{x}{2} - \cot x$.
Then
\begin{align*} S &= \sum_{k=3}^{2019} \csc (2^k)^\circ = \sum_{k=3}^{2019} (\cot (2^{k-1})^\circ - \cot (2^k)^\circ) \\ &= (\cot (2^2)^\circ - \cot (2^3)^\circ) + (\cot (2^3)^\circ - \cot (2^4)^\circ) + \dots + (\cot (2^{2018})^\circ - \cot (2^{2019})^\circ) \\ &= \cot (2^2)^\circ - \cot (2^{2019})^\circ \\ &= \cot 4^\circ - \cot (2^{2019})^\circ. \end{align*}
We are given that $S = \sec n^\circ = \frac{1}{\cos n^\circ}$.
Thus, $\cot 4^\circ - \cot (2^{2019})^\circ = \frac{1}{\cos n^\circ}$.


In [35]:
sum([ea == '' for ea in results_llama_8['data']['extracted_answers']])

760

In [24]:
for judge in [results_llama_8,results_qwen_06, results_qwen_4, results_qwen_8]:
    for ref in [results_ds_distill_qwen, results_deepseek_r1, results_gpt_5]:
        diff_data_new = produce_diffed_data(judge, ref)
        print(diff_data_new['metadata'])

Error processing completion for model deepseek-ai/DeepSeek-R1-Distill-Qwen-32B at index 0: substring not found
Completion text: <｜begin▁of▁sentence｜>You are an expert on answering and explaining math questions. Read the question and formulate a response. Please reason step by step.
Formatting instructions:
1. When you have a final answer, you will wrap the final answer around a \boxed{} function so that it can be evaluated. If no such answer is found, you will receive no credit regardless of how much reasoning you did. 
2. Your answer should exclusively use standard LaTeX math formatting (e.g., \frac{a}{b} for fractions, \sqrt{x} for square roots, \infty for infinity etc.). Do not use any math symbols or characters other than LaTeX formats -- they are ugly and indecipherable.
3. Do not include any decorators like dollar signs ($) in your final answer.
4. Do not surround the answer in a LaTeX code block. Return it as-is.
5. Do not include any units, suffixes, or variable definitions unl

UnboundLocalError: local variable 'completions_2_clean' referenced before assignment

In [ ]:
diff_data

In [97]:
ask = [k for k in results_2['data']['completions']  if "</think>" not in k and "<think>" in k]

In [115]:
len(ask[10])

6111